# **Limpieza y preparación de datos — Calidad del aire (Año 2024)**

## Descripción del dataset
En esta sección se realiza la limpieza y preparación de los datos correspondientes a la calidad del aire del año 2024, específicamente de la hoja **"agu"**, la cual contiene mediciones horarias de distintos contaminantes atmosféricos.

Las variables de interés para este análisis son:
- O3 (Ozono)
- PM10 (Partículas menores a 10 micras)
- PM2.5 (Partículas menores a 2.5 micras)

Además, se cuenta con información temporal:
- Fecha
- Hora
---

### **1. Carga de datos**

se importaron las librerias necesarias y se cargaron los datos de **`2024`** de la pagina **`AGU`**



In [2]:
import pandas as pd
import numpy as np

df_agu = pd.read_excel("/Users/heribertovs/Library/CloudStorage/OneDrive-ITESO/Semestre 6/Programacion para Mineria De Datos/proyecto_mineria_datos/data/raw/datos_2024.xlsx", sheet_name="AGU")



### **2. Exploración inicial de los datos**

Se realizó una inspección inicial del dataset utilizando:

>```python
>df_agu.info()
>df_agu.isnull().sum()

In [3]:

df_agu.info()
df_agu.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Fecha    8784 non-null   datetime64[us]
 1   Hora     8784 non-null   int64         
 2   O3       8784 non-null   object        
 3   NO       8784 non-null   object        
 4   NO2      8784 non-null   object        
 5   NOX      8784 non-null   object        
 6   SO2      8784 non-null   str           
 7   CO       8784 non-null   object        
 8   PM10     8784 non-null   object        
 9   PM2.5    8784 non-null   object        
 10  TempInt  8784 non-null   object        
 11  TempExt  8784 non-null   object        
 12  RH       8784 non-null   object        
 13  WS       8784 non-null   object        
 14  WD       8784 non-null   object        
 15  RS       8784 non-null   str           
 16  IUV      8784 non-null   str           
 17  PP       8784 non-null   object        
 18 

Fecha      0
Hora       0
O3         0
NO         0
NO2        0
NOX        0
SO2        0
CO         0
PM10       0
PM2.5      0
TempInt    0
TempExt    0
RH         0
WS         0
WD         0
RS         0
IUV        0
PP         0
Presion    0
dtype: int64

### **3. Tratamiento de valores inválidos**

Durante la exploración del dataset se identificó que no existían valores nulos explícitos, pero sí múltiples registros con banderas que representan datos inválidos o faltantes.

Las banderas identificadas fueron:
- `SE`: sin equipo
- `IF`: inválido por falla
- `IO`: inválido por operador
- `IR`: inválido por fuera de rango
- `ND`: sin dato
- `VE`: valor extraordinario
- `NE`: no existía la estación en esa fecha
- `IC`: inválido por calibración

Para poder analizarlos correctamente, estas banderas fueron reemplazadas por valores `NaN`:

>```python
>flags = ["SE", "IF", "IO", "IR", "ND", "VE", "NE", "IC"]
>df_agu = df_agu.replace(flags, np.nan)

In [4]:


flags = ["SE", "IF", "IO", "IR", "ND", "VE", "NE", "IC"]

df_agu = df_agu.replace(flags, np.nan)

df_agu.info()
df_agu.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 19 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Fecha    8784 non-null   datetime64[us]
 1   Hora     8784 non-null   int64         
 2   O3       8433 non-null   object        
 3   NO       2705 non-null   object        
 4   NO2      2702 non-null   object        
 5   NOX      2705 non-null   object        
 6   SO2      0 non-null      str           
 7   CO       1442 non-null   object        
 8   PM10     8577 non-null   object        
 9   PM2.5    7764 non-null   object        
 10  TempInt  7871 non-null   object        
 11  TempExt  7865 non-null   object        
 12  RH       4727 non-null   object        
 13  WS       4547 non-null   object        
 14  WD       4633 non-null   object        
 15  RS       0 non-null      str           
 16  IUV      0 non-null      str           
 17  PP       4389 non-null   object        
 18 

Fecha         0
Hora          0
O3          351
NO         6079
NO2        6082
NOX        6079
SO2        8784
CO         7342
PM10        207
PM2.5      1020
TempInt     913
TempExt     919
RH         4057
WS         4237
WD         4151
RS         8784
IUV        8784
PP         4395
Presion    6231
dtype: int64

### **3.5. Tratamiento de valores inválidos**

Tras el reemplazo de las banderas por valores nulos, se observó que varias variables presentan una cantidad significativa de datos faltantes. 

En particular, contaminantes como `NO, NO2, NOX, CO` y Presión muestran altos niveles de ausencia de información, lo que limita su utilidad para el análisis en esta etapa. Por otro lado, variables como `O3, PM10 y PM2.5` presentan una menor proporción de valores nulos, por lo que se consideran más adecuadas para continuar con el estudio.

Debido a lo anterior, se decidió enfocar el análisis en estas tres variables, ya que permiten obtener resultados más confiables y representativos sin requerir técnicas complejas de imputación en esta fase del proyecto.

### **4. Transformación de variables**

En esta etapa se realizaron las transformaciones necesarias para asegurar que los datos fueran consistentes y adecuados para el análisis.

### Conversión de variables a tipo numérico

Las variables de contaminantes (O3, PM10 y PM2.5) se encontraban en formato texto, por lo que se convirtieron a tipo numérico utilizando la función `pd.to_numeric()`.

Los valores no convertibles fueron transformados a `NaN` mediante el parámetro `errors='coerce'`.

### Construcción de la variable temporal

Se creó una nueva variable `datetime` combinando la fecha y la hora, con el objetivo de facilitar el análisis de series de tiempo.

Para ello:
- Se normalizó la columna de fecha eliminando cualquier componente de hora.
- Se sumó la variable `Hora` como un desplazamiento en horas.

### Selección de variables relevantes

Finalmente, se creó un nuevo dataset que contiene únicamente las variables necesarias para el análisis:
- datetime
- O3
- PM10
- PM2.5

Esto permite trabajar con un conjunto de datos más limpio y enfocado.

In [5]:
cols_numericas = ["O3", "PM10", "PM2.5"]

for col in cols_numericas:
    df_agu[col] = pd.to_numeric(df_agu[col], errors='coerce')


df_agu["Fecha "] = pd.to_datetime(df_agu["Fecha "]).dt.normalize()
df_agu["datetime"] = df_agu["Fecha "] + pd.to_timedelta(df_agu["Hora"], unit="h")
df_agu["id_estacion"] = "AGU"
df_agu["estacion"] = "Aguilas"

df_agu_limpio = df_agu[["id_estacion", "estacion","datetime", "O3", "PM10", "PM2.5"]]

### **5. Exportación del dataset limpio**

Una vez finalizado el proceso de limpieza y transformación, se generó un nuevo dataset con las variables relevantes (`datetime`, `O3`, `PM10`, `PM2.5`) y se exportó en formato CSV.

El archivo fue almacenado en la carpeta **interim**, la cual contiene datos parcialmente procesados que han pasado por una etapa inicial de limpieza y preparación.

>```python
>df_agu_limpio.to_csv("ruta_del_archivo.csv", index=False)
>

In [12]:
df_agu_limpio.to_csv("/Users/heribertovs/Library/CloudStorage/OneDrive-ITESO/Semestre 6/Programacion para Mineria De Datos/proyecto_mineria_datos/data/interim/agu_2024_conNulos.csv", index=False)

In [11]:
df_agu_limpio.info()
df_agu_limpio.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_estacion  8784 non-null   str           
 1   estacion     8784 non-null   str           
 2   datetime     8784 non-null   datetime64[us]
 3   O3           8433 non-null   float64       
 4   PM10         8577 non-null   float64       
 5   PM2.5        7764 non-null   float64       
dtypes: datetime64[us](1), float64(3), str(2)
memory usage: 411.9 KB


id_estacion       0
estacion          0
datetime          0
O3              351
PM10            207
PM2.5          1020
dtype: int64

In [13]:
df_agu_na = df_agu_limpio.dropna()

In [14]:
df_agu_na.to_csv("/Users/heribertovs/Library/CloudStorage/OneDrive-ITESO/Semestre 6/Programacion para Mineria De Datos/proyecto_mineria_datos/data/interim/agu_2024_sinNulos.csv", index=False)

In [16]:
df_agu_na.info()
df_agu_na.isnull().sum()

<class 'pandas.DataFrame'>
Index: 7507 entries, 0 to 8783
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id_estacion  7507 non-null   str           
 1   estacion     7507 non-null   str           
 2   datetime     7507 non-null   datetime64[us]
 3   O3           7507 non-null   float64       
 4   PM10         7507 non-null   float64       
 5   PM2.5        7507 non-null   float64       
dtypes: datetime64[us](1), float64(3), str(2)
memory usage: 410.5 KB


id_estacion    0
estacion       0
datetime       0
O3             0
PM10           0
PM2.5          0
dtype: int64